# [6.3] Transcoders and Attribution Graphs - Exercises

Transcoders are sparse replacements for model components. In this notebook you build the mechanics behind that claim: run a ReLU transcoder, check whether replacement preserves logits, turn features into attribution edges, and test whether selected graph nodes matter more than low-effect controls.

```yaml
gt_tier: GT-0/GT-1 real-model preflight
exercise_id: 6_3_transcoders_and_attribution_graphs
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA GELU-1L preflight
requires_gpu: true for the TransformerLens preflight; false for the implementation exercises
```

<details>
<summary>Expected output</summary>

By the end, every local test should print an "All tests ... passed" line, and the final report-backed cells should show the pinned `gelu-1l` oracle replacement, trained tiny transcoder, graph-control, and VRAM metrics.

</details>

<details>
<summary>Help - how to read this section</summary>

The sequence is replacement first, interpretation second. Do not trust a graph because its nodes have plausible names; trust it only to the extent that replacement and damage controls survive.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part3_transcoders_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_transcoders_attribution_graphs.tests as tests


@dataclass(frozen=True)
class TranscoderOutput:
    feature_acts: t.Tensor
    reconstructed_activations: t.Tensor


@dataclass(frozen=True)
class TranscoderReplacementReport:
    reconstruction_mse: float
    replacement_kl: float
    target_logit_diff: float
    replacement_logit_diff: float
    logit_diff_error: float
    passes_kl: bool
    preserves_logit_diff: bool


@dataclass(frozen=True)
class AttributionEdge:
    source_type: str
    source_id: int
    target_type: str
    target_id: int
    weight: float


@dataclass(frozen=True)
class AttributionGraphReport:
    num_nodes: int
    num_edges: int
    density: float
    full_logit_diff: float
    graph_logit_diff: float
    ablated_logit_diff: float
    random_ablated_logit_diff: float
    topk_damage: float
    random_damage: float
    preserves_logit_diff: bool
    passes_damage_control: bool
    reproducible: bool


## 1. Transcoder Forward

A one-hidden-layer transcoder maps MLP inputs into sparse feature activations and decodes those activations into the MLP-output space.

<details>
<summary>Expected output</summary>

The test checks both an identity reconstruction and a biased example where negative encoder pre-activations are clamped to zero. When correct, it prints:

```text
All tests in `test_transcoder_forward_matches_reference_and_relu_rules` passed!
```

</details>

<details>
<summary>Help - the replacement has two separate maps</summary>

The encoder finds sparse features; the decoder maps feature activations back into the component-output space. If you mix up the directions, the replacement no longer interfaces with the model component you mean to replace.

</details>


In [ ]:
def transcoder_forward(
    inputs: t.Tensor,
    encoder_weight: t.Tensor,
    decoder_weight: t.Tensor,
    *,
    encoder_bias: t.Tensor | None = None,
    decoder_bias: t.Tensor | None = None,
) -> TranscoderOutput:
    raise NotImplementedError()


tests.test_transcoder_forward_matches_reference_and_relu_rules(transcoder_forward)


## 2. Replacement Metrics

A good-looking reconstruction can still change the output distribution. Here you implement mean KL, target logit difference, and a replacement report that checks both behavior and reconstruction.

<details>
<summary>Expected output</summary>

The controlled replacement should pass KL and logit-diff preservation, and the target logit difference should be the batch mean. When correct, it prints:

```text
All tests in `test_target_logit_diff_and_replacement_report_match_reference` passed!
```

</details>

<details>
<summary>Help - reconstruction is not behavior</summary>

MSE is about hidden states; KL and logit differences are about model behavior. A replacement claim needs both views, because the downstream unembedding can amplify small hidden-state errors.

</details>


In [ ]:
def mean_kl_divergence(reference_logits: t.Tensor, reconstructed_logits: t.Tensor) -> float:
    raise NotImplementedError()


def target_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


def transcoder_replacement_report(
    *,
    reference_activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    reference_logits: t.Tensor,
    replacement_logits: t.Tensor,
    positive_token_id: int,
    negative_token_id: int,
    kl_threshold: float = 1e-3,
    logit_diff_tolerance: float = 0.1,
) -> TranscoderReplacementReport:
    raise NotImplementedError()


tests.test_target_logit_diff_and_replacement_report_match_reference(
    target_logit_diff,
    transcoder_replacement_report,
)


## 3. Feature Contributions

For a target logit difference, multiply each feature's mean activation by its direct logit effect. This gives graph edge weights to test, not a finished circuit explanation.

<details>
<summary>Expected output</summary>

The controlled rank-3 tensor should reduce over batch and sequence dimensions and return `[0.75, 2.0, -1.0]`. When correct, it prints:

```text
All tests in `test_feature_logit_contributions_reduce_all_nonfeature_dimensions` passed!
```

</details>

<details>
<summary>Help - contributions are graph hypotheses</summary>

A large contribution tells you where to look. It does not prove causality until an intervention or contribution-removal control checks whether removing those features damages the target metric.

</details>


In [ ]:
def feature_logit_contributions(feature_acts: t.Tensor, logit_effects: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_feature_logit_contributions_reduce_all_nonfeature_dimensions(
    feature_logit_contributions,
)


## 4. Graph Edges

Build a sparse graph in two blocks: input-to-feature edges and feature-to-logit-difference edges. Choose edges by absolute magnitude but keep signed weights.

<details>
<summary>Expected output</summary>

Top input-feature and feature-logit edges should match the reference edge list, and repeated construction should be reproducible. When correct, the tests print:

```text
All tests in `test_build_attribution_edges_keeps_top_input_and_logit_edges` passed!
All tests in `test_graph_reproducible_rejects_structure_and_weight_changes` passed!
```

</details>

<details>
<summary>Help - signed negative edges can matter</summary>

Sort by absolute value so large inhibitory edges survive selection, but store the original sign so the graph still knows whether the feature pushes toward or away from the target.

</details>


In [ ]:
def build_attribution_edges(
    input_to_feature_scores: t.Tensor,
    feature_logit_effects: t.Tensor,
    *,
    top_k: int,
) -> list[AttributionEdge]:
    raise NotImplementedError()


def graph_reproducible(
    edges_a: list[AttributionEdge],
    edges_b: list[AttributionEdge],
    *,
    atol: float = 1e-6,
) -> bool:
    raise NotImplementedError()


tests.test_build_attribution_edges_keeps_top_input_and_logit_edges(
    build_attribution_edges,
    graph_reproducible,
)
tests.test_graph_reproducible_rejects_structure_and_weight_changes(
    AttributionEdge,
    graph_reproducible,
)


## 5. Graph Reports

The graph report asks whether selected features explain enough of the target logit difference and whether removing them damages the target more than ablating low-effect controls.

<details>
<summary>Expected output</summary>

Selecting features `[0, 1]` from `[0.7, 0.2, 0.05, 0.05]` should preserve the logit difference and beat the low-effect control. Selecting `[2, 3]` should fail. When correct, it prints:

```text
All tests in `test_attribution_graph_report_preservation_and_damage_controls` passed!
```

</details>

<details>
<summary>Help - a graph needs a damage control</summary>

A graph selected from contribution scores can look good by construction. The control is to remove low-effect features under the same rule and check that they damage the metric much less.

</details>


In [ ]:
def graph_density(*, num_nodes: int, num_edges: int) -> float:
    raise NotImplementedError()


def attribution_graph_report(
    contributions: t.Tensor,
    graph_feature_ids: t.Tensor | list[int],
    random_feature_ids: t.Tensor | list[int],
    *,
    num_nodes: int,
    num_edges: int,
    reproducible: bool,
    preservation_threshold: float = 0.8,
) -> AttributionGraphReport:
    raise NotImplementedError()


tests.test_attribution_graph_report_preservation_and_damage_controls(
    graph_density,
    attribution_graph_report,
)


## Whole-Notebook Contract

Once all exercises pass, your implementation should satisfy the same local smoke-test contract as `solutions.py`.

<details>
<summary>Expected output</summary>

After uncommenting the last line, the test should print:

```text
All tests in `test_notebook_contract` passed!
```

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    forward = transcoder_forward(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.eye(2), t.eye(2))
    replacement = transcoder_replacement_report(
        reference_activations=t.tensor([[1.0, 2.0]]),
        reconstructed_activations=t.tensor([[1.01, 2.01]]),
        reference_logits=t.tensor([[2.0, 0.0, -1.0]]),
        replacement_logits=t.tensor([[1.98, 0.02, -1.0]]),
        positive_token_id=0,
        negative_token_id=1,
        kl_threshold=1e-3,
        logit_diff_tolerance=0.1,
    )
    edges = build_attribution_edges(
        t.tensor([[0.1, 0.8], [0.4, 0.2]]),
        t.tensor([0.3, 1.0]),
        top_k=1,
    )
    repeated = build_attribution_edges(
        t.tensor([[0.1, 0.8], [0.4, 0.2]]),
        t.tensor([0.3, 1.0]),
        top_k=1,
    )
    return {
        "forward": {
            "feature_acts": forward.feature_acts.tolist(),
            "reconstructed_activations": forward.reconstructed_activations.tolist(),
        },
        "replacement": replacement.__dict__,
        "contributions": {
            "contributions": feature_logit_contributions(
                t.tensor([[1.0, 2.0, 0.0], [3.0, 0.0, 2.0]]),
                t.tensor([0.5, 1.0, -1.0]),
            ).tolist(),
        },
        "graph_edges": {"reproducible": graph_reproducible(edges, repeated)},
        "graph_report": attribution_graph_report(
            t.tensor([0.7, 0.2, 0.05, 0.05]),
            graph_feature_ids=[0, 1],
            random_feature_ids=[2, 3],
            num_nodes=6,
            num_edges=4,
            reproducible=True,
        ).__dict__,
    }


# Uncomment after finishing all exercises.
# tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final result is report-backed and uses the committed CUDA evidence. It does not rerun the TransformerLens path inside the notebook; rerun `solutions.run_gpu_test(max_vram_gb=24.0)` from Python when you want to refresh the report.

<details>
<summary>Expected output</summary>

The table should show `gelu-1l`, `104` cached activations, oracle MLP/logit max errors near `1e-5`, tiny transcoder held-out MSE ratio around `0.150`, top-1 agreement around `0.904`, graph damage `0.808` vs `0.000646`, and peak VRAM under `1 GB`.

</details>

<details>
<summary>Interpreting the signature result</summary>

This proves a scoped GELU-1L preflight: exact oracle replacement, a trained tiny ReLU transcoder that beats a zero-output baseline, and a top-feature graph that beats a low-effect damage control. It is not a published frontier-transcoder replication.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", gpu["model_name"]),
        ("HF revision", gpu["hf_revision"][:12]),
        ("prompts / activations", f"{gpu['prompt_count']} / {gpu['activation_count']}"),
        ("d_model / MLP width", f"{gpu['d_model']} / {gpu['mlp_width']}"),
        ("oracle MLP-out max error", f"{gpu['oracle_mlp_out_max_abs_error']:.2e}"),
        ("oracle logits max error", f"{gpu['oracle_logits_max_abs_error']:.2e}"),
        ("tiny transcoder width / steps", f"{gpu['trained_transcoder_width']} / {gpu['trained_transcoder_steps']}"),
        ("held-out MSE / zero MSE", f"{gpu['trained_transcoder_heldout_mse']:.3f} / {gpu['trained_transcoder_heldout_zero_mse']:.3f}"),
        ("held-out MSE ratio", round(gpu["trained_transcoder_heldout_mse_ratio"], 3)),
        ("top-1 agreement", round(gpu["trained_replacement_top1_agreement"], 3)),
        ("feature density", round(gpu["trained_transcoder_feature_density"], 3)),
        ("graph top / low-effect damage", f"{gpu['graph_topk_damage']:.3f} / {gpu['graph_random_damage']:.6f}"),
        ("graph reproducible", gpu["graph_reproducible"]),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["MLP out", "logits"],
    [gpu["oracle_mlp_out_max_abs_error"], gpu["oracle_logits_max_abs_error"]],
    color=["#2563eb", "#0f766e"],
)
axes[0].set_title("Oracle replacement errors")
axes[0].set_ylabel("max abs error")

axes[1].bar(
    ["trained", "zero baseline"],
    [gpu["trained_transcoder_heldout_mse"], gpu["trained_transcoder_heldout_zero_mse"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title("Held-out reconstruction")
axes[1].set_ylabel("MSE, lower is better")

axes[2].bar(
    ["top graph", "low-effect"],
    [gpu["graph_topk_damage"], gpu["graph_random_damage"]],
    color=["#7c3aed", "#f97316"],
)
axes[2].set_title("Graph damage control")
axes[2].set_ylabel("target logit-diff damage")

fig.tight_layout()
plt.show()


## Limitations

The local tests use tiny tensors. The CUDA report uses one small public TransformerLens checkpoint, safe generated prompts, and a tiny trained ReLU transcoder. It does not claim a production transcoder artifact, a frontier attribution graph replication, generated-completion behavior, or a complete causal circuit audit.
